# Mangrove benefit-cost ratio

This notebook builds **patch-level benefit-cost ratio (BCR)** tables for Jamaica mangroves using the current **weighted area-distance attribution plus nearest-neighbour fallback beyond 5000 m** results.

For each mangrove patch, the notebook compiles:
- `mangrove_patch_id`
- `area_ha`
- `EADs_baseline_year`
- `EADs_50_years`
- `patch_cost_baseline`
- `patch_cost_50_years`
- `bcr_cost_eads_baseline`
- `bcr_cost_eads_50_years`

The annual protection cost per hectare is taken from `protected_area_size.ipynb`, and the 50-year discounting uses the **same 10% discount-factor method** used elsewhere in the project and in the river flooding analysis.

`EADs_baseline_year` and `EADs_50_years` use **net attributed avoided EADs** at mangrove-patch level. Negative values therefore indicate patches with net increased damages rather than net protection.


In [ ]:
from pathlib import Path

import geopandas as geopandas
import numpy as np
import pandas as pd
from IPython.display import display

pd.options.display.float_format = "{:,.2f}".format

base_path = Path("/Users/robynhaggis/Documents/Geospatial_analysis/dphil_papers")
mangrove_path = base_path / "dphil_paper_3/inputs/forces_of_nature_mangroves/mangroves.shp"
protected_area_cost_csv = (
    base_path
    / "dphil_paper_3/processed_data/baseline_assessment/protected_area_size/protected_area_forest_reserve_usd7m_density.csv"
)
output_dir = base_path / "dphil_paper_3/results/06_cost_benefit_analysis/bcr_mangroves"
output_dir.mkdir(parents=True, exist_ok=True)

discount_years = 50
discount_rate = 0.10
discounted_suffix = "PV_50Y_10pct"

scenario_config = {
    "minimum": {
        "annual_patch_ead_csv": (
            base_path
            / "dphil_paper_3/results/02_damage_estimates/coastal_flood_damages/results_coastal_minimum_scenario/damage_estimates/mangrove_attribution_area_distance_all_sectors_signed/mangrove_attribution_total_all_sectors_signed_area_distance_5000m_nn_fallback.csv"
        ),
        "discounted_patch_ead_csv": (
            base_path
            / "dphil_paper_3/results/02_damage_estimates/coastal_flood_damages/results_coastal_minimum_scenario/damage_estimates/discounted_50_year_weighted_area_distance/mangrove_attribution_total_all_sectors_signed_area_distance_5000m_nn_fallback_50yr_discounted_10pct.csv"
        ),
    },
    "maximum": {
        "annual_patch_ead_csv": (
            base_path
            / "dphil_paper_3/results/02_damage_estimates/coastal_flood_damages/results_coastal_maximum_scenario/damage_estimates/mangrove_attribution_area_distance_all_sectors_signed/mangrove_attribution_total_all_sectors_signed_area_distance_5000m_nn_fallback.csv"
        ),
        "discounted_patch_ead_csv": (
            base_path
            / "dphil_paper_3/results/02_damage_estimates/coastal_flood_damages/results_coastal_maximum_scenario/damage_estimates/discounted_50_year_weighted_area_distance/mangrove_attribution_total_all_sectors_signed_area_distance_5000m_nn_fallback_50yr_discounted_10pct.csv"
        ),
    },
}


## Inputs and method

- Mangrove patch areas are taken from the same `mangroves.shp` layer used in the attribution notebooks.
- Patch-level benefits are taken from the existing patch attribution outputs for the **minimum** and **maximum** scenarios.
- The annual cost per hectare comes from the protected-area budget density table produced by `protected_area_size.ipynb`.
- The 50-year discounted values use the same formula applied elsewhere in the project:

\[
\sum_{y=0}^{50} \frac{1}{(1+r)^y}
\]

with `r = 0.10`.

Because both the annual benefits and the annual costs are multiplied by the same discount factor, the baseline-year and 50-year BCRs are expected to be numerically identical under this constant-annual-flow setup.


In [ ]:
def compute_discount_rate(years, discount_rate):
    discount_rates = [1 / (1 + discount_rate) ** year for year in range(years + 1)]
    discount_rates_sum = sum(discount_rates)
    return discount_rates_sum


def load_mangrove_patch_areas(mangrove_path):
    mangrove_patches = geopandas.read_file(mangrove_path)
    if mangrove_patches.crs is None:
        raise ValueError("Mangrove CRS is missing.")
    if str(mangrove_patches.crs).upper() != "EPSG:3448":
        mangrove_patches = mangrove_patches.to_crs("EPSG:3448")

    if "ID" in mangrove_patches.columns:
        mangrove_patches["mangrove_patch_id"] = mangrove_patches["ID"].astype(int)
    else:
        mangrove_patches = mangrove_patches.reset_index(drop=True)
        mangrove_patches["mangrove_patch_id"] = np.arange(1, len(mangrove_patches) + 1)

    mangrove_patches["area_ha"] = mangrove_patches.geometry.area / 10_000.0

    patch_area_table = (
        mangrove_patches[["mangrove_patch_id", "area_ha"]]
        .drop_duplicates(subset=["mangrove_patch_id"])
        .sort_values("mangrove_patch_id")
        .reset_index(drop=True)
    )
    return patch_area_table


def build_patch_bcr_table(
    patch_area_table,
    annual_patch_ead_csv,
    discounted_patch_ead_csv,
    cost_per_ha_usd,
    discount_factor,
):
    annual_patch_ead = pd.read_csv(annual_patch_ead_csv)[
        ["Mangrove_ID", "Net_Avoided_EAD_USD_attributed"]
    ].rename(
        columns={
            "Mangrove_ID": "mangrove_patch_id",
            "Net_Avoided_EAD_USD_attributed": "EADs_baseline_year",
        }
    )

    discounted_patch_ead = pd.read_csv(discounted_patch_ead_csv)[
        ["Mangrove_ID", f"Net_Avoided_EAD_USD_attributed_{discounted_suffix}"]
    ].rename(
        columns={
            "Mangrove_ID": "mangrove_patch_id",
            f"Net_Avoided_EAD_USD_attributed_{discounted_suffix}": "EADs_50_years",
        }
    )

    patch_bcr_table = (
        patch_area_table
        .merge(annual_patch_ead, on="mangrove_patch_id", how="left")
        .merge(discounted_patch_ead, on="mangrove_patch_id", how="left")
    )

    for ead_column in ["EADs_baseline_year", "EADs_50_years"]:
        patch_bcr_table[ead_column] = pd.to_numeric(
            patch_bcr_table[ead_column], errors="coerce"
        ).fillna(0.0)

    patch_bcr_table["patch_cost_baseline"] = patch_bcr_table["area_ha"] * cost_per_ha_usd
    patch_bcr_table["patch_cost_50_years"] = patch_bcr_table["patch_cost_baseline"] * discount_factor

    patch_bcr_table["bcr_cost_eads_baseline"] = np.where(
        patch_bcr_table["patch_cost_baseline"] > 0,
        patch_bcr_table["EADs_baseline_year"] / patch_bcr_table["patch_cost_baseline"],
        np.nan,
    )
    patch_bcr_table["bcr_cost_eads_50_years"] = np.where(
        patch_bcr_table["patch_cost_50_years"] > 0,
        patch_bcr_table["EADs_50_years"] / patch_bcr_table["patch_cost_50_years"],
        np.nan,
    )

    patch_bcr_table = patch_bcr_table[
        [
            "mangrove_patch_id",
            "area_ha",
            "EADs_baseline_year",
            "EADs_50_years",
            "patch_cost_baseline",
            "patch_cost_50_years",
            "bcr_cost_eads_baseline",
            "bcr_cost_eads_50_years",
        ]
    ].sort_values(
        ["bcr_cost_eads_baseline", "EADs_baseline_year", "mangrove_patch_id"],
        ascending=[False, False, True],
    ).reset_index(drop=True)

    return patch_bcr_table


def build_scenario_summary_table(patch_bcr_table, scenario_name):
    total_baseline_eads = float(patch_bcr_table["EADs_baseline_year"].sum())
    total_50_year_eads = float(patch_bcr_table["EADs_50_years"].sum())
    total_baseline_cost = float(patch_bcr_table["patch_cost_baseline"].sum())
    total_50_year_cost = float(patch_bcr_table["patch_cost_50_years"].sum())

    return {
        "scenario": scenario_name,
        "mangrove_patch_count": int(len(patch_bcr_table)),
        "zero_ead_patch_count": int((patch_bcr_table["EADs_baseline_year"] == 0).sum()),
        "negative_ead_patch_count": int((patch_bcr_table["EADs_baseline_year"] < 0).sum()),
        "total_EADs_baseline_year": total_baseline_eads,
        "total_EADs_50_years": total_50_year_eads,
        "total_patch_cost_baseline": total_baseline_cost,
        "total_patch_cost_50_years": total_50_year_cost,
        "national_bcr_cost_eads_baseline": total_baseline_eads / total_baseline_cost,
        "national_bcr_cost_eads_50_years": total_50_year_eads / total_50_year_cost,
    }


In [ ]:
discount_factor_50_years = compute_discount_rate(discount_years, discount_rate)
protected_area_cost_table = pd.read_csv(protected_area_cost_csv)
cost_per_ha_usd = float(protected_area_cost_table.loc[0, "usd_per_ha"])
patch_area_table = load_mangrove_patch_areas(mangrove_path)

print(f"Discount factor ({discount_years} years, {discount_rate:.0%}) = {discount_factor_50_years:,.12f}")
print(f"Annual protection cost per hectare (USD/ha/year) = {cost_per_ha_usd:,.2f}")
print(f"Mangrove patches in source layer = {len(patch_area_table)}")

display(protected_area_cost_table)
display(patch_area_table.head())


In [ ]:
scenario_patch_bcr_tables = {}
scenario_summary_rows = []
saved_csv_paths = []

for scenario_name, scenario_paths in scenario_config.items():
    patch_bcr_table = build_patch_bcr_table(
        patch_area_table=patch_area_table,
        annual_patch_ead_csv=scenario_paths["annual_patch_ead_csv"],
        discounted_patch_ead_csv=scenario_paths["discounted_patch_ead_csv"],
        cost_per_ha_usd=cost_per_ha_usd,
        discount_factor=discount_factor_50_years,
    )
    scenario_patch_bcr_tables[scenario_name] = patch_bcr_table
    scenario_summary_rows.append(build_scenario_summary_table(patch_bcr_table, scenario_name))

    scenario_output_csv = output_dir / f"mangrove_patch_bcr_{scenario_name}_scenario.csv"
    patch_bcr_table.to_csv(scenario_output_csv, index=False)
    saved_csv_paths.append(scenario_output_csv)

    print(f"Saved {scenario_name} scenario patch BCR table: {scenario_output_csv}")
    display(patch_bcr_table.head(15))

scenario_summary_table = pd.DataFrame(scenario_summary_rows).sort_values("scenario").reset_index(drop=True)
comparison_table = pd.concat(
    [
        patch_bcr_table.assign(scenario=scenario_name)
        for scenario_name, patch_bcr_table in scenario_patch_bcr_tables.items()
    ],
    ignore_index=True,
)
comparison_table = comparison_table[
    [
        "scenario",
        "mangrove_patch_id",
        "area_ha",
        "EADs_baseline_year",
        "EADs_50_years",
        "patch_cost_baseline",
        "patch_cost_50_years",
        "bcr_cost_eads_baseline",
        "bcr_cost_eads_50_years",
    ]
].sort_values(
    ["scenario", "bcr_cost_eads_baseline", "EADs_baseline_year", "mangrove_patch_id"],
    ascending=[True, False, False, True],
).reset_index(drop=True)

scenario_summary_csv = output_dir / "mangrove_patch_bcr_summary.csv"
comparison_csv = output_dir / "mangrove_patch_bcr_minimum_maximum_comparison.csv"
scenario_summary_table.to_csv(scenario_summary_csv, index=False)
comparison_table.to_csv(comparison_csv, index=False)
saved_csv_paths.extend([scenario_summary_csv, comparison_csv])

print("Saved summary outputs:")
for saved_csv_path in saved_csv_paths:
    print(f" - {saved_csv_path}")

print("Scenario summary:")
display(scenario_summary_table)


In [ ]:
bcr_consistency_rows = []
for scenario_name, patch_bcr_table in scenario_patch_bcr_tables.items():
    bcr_difference = (
        patch_bcr_table["bcr_cost_eads_baseline"] - patch_bcr_table["bcr_cost_eads_50_years"]
    ).abs()
    bcr_consistency_rows.append(
        {
            "scenario": scenario_name,
            "max_abs_difference_between_bcr_columns": float(bcr_difference.max()),
            "mean_abs_difference_between_bcr_columns": float(bcr_difference.mean()),
        }
    )

bcr_consistency_table = pd.DataFrame(bcr_consistency_rows).sort_values("scenario").reset_index(drop=True)
print("BCR consistency check:")
display(bcr_consistency_table)
